## SQL 조회 + 데이터 분석 Multi-Agent System

이번 실습에서는 **두 개의 전문 에이전트**를 조합해 사내 DB 데이터를 조회하고 분석하는 멀티 에이전트 시스템을 구축합니다.

```
사용자 질문
     ↓
Supervisor Agent  ← 질문을 보고 알맞은 sub-agent에게 전달
     ├── DB 조회 Agent    : SQL을 작성해 사내 DB에서 데이터 추출
     └── 데이터 분석 Agent : pandas로 데이터 통계·집계 분석
```

### 핵심 포인트

**왜 두 에이전트로 분리하나요?**

하나의 에이전트가 SQL 작성과 데이터 분석을 동시에 담당하면, LLM이 직접 숫자를 계산하거나 집계해야 하는 상황이 생깁니다.
LLM의 숫자 연산은 부정확할 수 있기 때문에, **SQL로 데이터를 가져오는 역할**과 **pandas로 정확하게 분석하는 역할**을 분리하는 것이 훨씬 안정적입니다.

| Sub-agent | 역할 |
|-----------|------|
| DB 조회 에이전트 | 사내 DB 접근 함수를 Tool로 감싸서 SQL 작성·실행 |
| 데이터 분석 에이전트 | DB 결과(JSON)를 pandas로 받아 통계·집계 수행 |

### 환경 설정

In [ ]:
import os
import json
import uuid
import sqlite3
import pandas as pd
from dotenv import load_dotenv, find_dotenv

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

load_dotenv(find_dotenv())

credential_key = os.getenv('credential_key')
send_system_name = os.getenv('send_system_name')
model_name = os.getenv('model')
api_base_url = os.getenv('api_base_url')
user_id = os.getenv('user_id')

os.environ['OPENAI_API_KEY'] = 'api_key'

llm = ChatOpenAI(
    model=model_name,
    base_url=api_base_url,
    default_headers={
        'x-dep-ticket': credential_key,
        'Send-System-Name': send_system_name,
        'User-Id': user_id,
        'User-Type': 'AD_ID',
        'Prompt-Msg-Id': str(uuid.uuid4()),
        'Completion-Msg-Id': str(uuid.uuid4())
    },
    temperature=0,
)

### 실습용 DB 준비

실제 사내 DB 환경을 시뮬레이션합니다.
실습에서는 SQLite 인메모리 DB를 사용하지만, 실제 환경에서는 아래 두 함수만 사내 코드로 교체하면 됩니다.

```python
# ↓ 실제 환경에서 교체할 부분
def run_query(sql: str) -> list[dict]:
    # 사내 DB 접근 로직 (인증, 프록시 등)
    ...

def fetch_schema(table_name: str) -> str:
    # 사내 방식으로 스키마 조회
    ...
```

나머지 Tool과 에이전트 코드는 수정 없이 그대로 사용할 수 있습니다.

In [ ]:
# 실습용 SQLite 인메모리 DB 생성
conn = sqlite3.connect(':memory:')

conn.execute('''
CREATE TABLE sales (
    sale_id    INTEGER PRIMARY KEY,
    product    TEXT,
    category   TEXT,
    amount     REAL,
    quantity   INTEGER,
    sale_month TEXT,
    region     TEXT
)
''')

sample_rows = [
    (1,  '노트북 Pro',    '전자기기', 1500000, 3,  '2024-01', '서울'),
    (2,  '무선 마우스',   '전자기기',   45000, 10, '2024-01', '부산'),
    (3,  '모니터 27인치', '전자기기',  380000, 5,  '2024-02', '서울'),
    (4,  '키보드',        '전자기기',   89000, 8,  '2024-02', '서울'),
    (5,  '책상',          '가구',      320000, 2,  '2024-01', '서울'),
    (6,  '의자',          '가구',      450000, 4,  '2024-02', '부산'),
    (7,  '노트북 Pro',    '전자기기', 1500000, 5,  '2024-03', '부산'),
    (8,  '무선 이어폰',   '전자기기',  230000, 12, '2024-03', '서울'),
    (9,  '책상',          '가구',      320000, 3,  '2024-03', '제주'),
    (10, '태블릿',        '전자기기',  890000, 6,  '2024-04', '서울'),
    (11, '의자',          '가구',      450000, 2,  '2024-04', '서울'),
    (12, '모니터 27인치', '전자기기',  380000, 4,  '2024-04', '제주'),
    (13, '노트북 Pro',    '전자기기', 1500000, 2,  '2024-05', '서울'),
    (14, '키보드',        '전자기기',   89000, 15, '2024-05', '부산'),
    (15, '책상',          '가구',      320000, 1,  '2024-05', '제주'),
]

conn.executemany('INSERT INTO sales VALUES (?,?,?,?,?,?,?)', sample_rows)
conn.commit()

print(f'DB 생성 완료 — {len(sample_rows)}건 삽입')

In [ ]:
# ↓ 실제 환경에서 이 두 함수를 사내 DB 접근 코드로 교체하세요

def run_query(sql: str) -> list:
    '''SQL을 실행하고 결과를 list[dict]로 반환합니다.'''
    cursor = conn.execute(sql)
    columns = [desc[0] for desc in cursor.description]
    return [dict(zip(columns, row)) for row in cursor.fetchall()]


def fetch_schema(table_name: str) -> str:
    '''테이블 스키마(컬럼명, 타입)를 반환합니다.'''
    cursor = conn.execute(f'PRAGMA table_info({table_name})')
    columns = cursor.fetchall()
    lines = [f'  {col[1]} ({col[2]})' for col in columns]
    return f'테이블: {table_name}\n컬럼:\n' + '\n'.join(lines)

### 1. DB 조회 에이전트 만들기

DB 조회 에이전트는 4개의 Tool을 사용합니다.

| Tool | 역할 |
|------|------|
| `get_table_list` | 조회 가능한 테이블 목록 반환 |
| `get_table_schema` | 특정 테이블의 컬럼 구조 반환 |
| `validate_query` | 쿼리 문법 검증 — 실제 실행 없이 |
| `execute_query` | 검증된 SELECT 쿼리 실행 |

#### 왜 Tool 레벨에서 안전 검증을 하나요?

LLM 프롬프트에 "SELECT만 써라"고 지시해도 완벽하지 않습니다.
**프롬프트 인젝션**이나 LLM의 실수 가능성이 있기 때문에, Tool 코드 자체에서 이중으로 검증하는 것이 필수입니다.

```
LLM이 쿼리 작성
     ↓
validate_query  ← SELECT 여부, 위험 키워드, 주석 차단
     ↓ (통과 시)
execute_query   ← 실행 전 한번 더 검사 + 결과 행 수 100개 제한
```

LLM을 신뢰하지 말고, **Tool이 최후의 방어선**이 되어야 합니다.

In [ ]:
# 접근 허용할 테이블과 금지 키워드 정의
ALLOWED_TABLES = ['sales']
FORBIDDEN_KEYWORDS = ['DROP', 'DELETE', 'UPDATE', 'INSERT', 'ALTER', 'TRUNCATE', 'CREATE', 'EXEC']


@tool
def get_table_list() -> str:
    '''조회 가능한 테이블 목록을 반환합니다.'''
    return '사용 가능한 테이블: ' + ', '.join(ALLOWED_TABLES)


@tool
def get_table_schema(table_name: str) -> str:
    '''특정 테이블의 컬럼 구조를 반환합니다.'''
    if table_name not in ALLOWED_TABLES:
        return f'Error: {table_name} 테이블에 접근할 수 없습니다.'
    return fetch_schema(table_name)


@tool
def validate_query(sql: str) -> str:
    '''SQL 쿼리를 실행 전에 검증합니다. 실제 데이터는 가져오지 않습니다.'''
    sql_upper = sql.upper().strip()

    if not sql_upper.startswith('SELECT'):
        return 'Error: SELECT 쿼리만 허용됩니다.'

    for kw in FORBIDDEN_KEYWORDS:
        if kw in sql_upper:
            return f'Error: {kw} 구문은 허용되지 않습니다.'

    if '--' in sql or '/*' in sql:
        return 'Error: SQL 주석은 허용되지 않습니다.'

    try:
        # EXPLAIN으로 실제 실행 없이 문법 오류만 검사
        conn.execute('EXPLAIN QUERY PLAN ' + sql)
        return '검증 통과. execute_query로 실행할 수 있습니다.'
    except Exception as e:
        return f'쿼리 오류: {e}'


@tool
def execute_query(sql: str) -> str:
    '''검증된 SELECT 쿼리를 실행하고 JSON 형식으로 결과를 반환합니다.'''
    sql_upper = sql.upper().strip()

    # execute_query에서도 이중 검증 — Tool은 항상 스스로 방어해야 합니다
    if not sql_upper.startswith('SELECT'):
        return 'Error: SELECT 쿼리만 허용됩니다.'

    for kw in FORBIDDEN_KEYWORDS:
        if kw in sql_upper:
            return f'Error: 허용되지 않는 구문 {kw}이 포함되어 있습니다.'

    if '--' in sql or '/*' in sql:
        return 'Error: SQL 주석은 허용되지 않습니다.'

    # 과도한 조회를 막기 위해 결과를 100개로 제한
    if 'LIMIT' not in sql_upper:
        sql = sql.rstrip(';') + ' LIMIT 100'

    try:
        results = run_query(sql)
        return json.dumps(results, ensure_ascii=False)
    except Exception as e:
        return f'쿼리 실행 오류: {e}'

In [ ]:
db_agent = create_agent(
    llm,
    tools=[get_table_list, get_table_schema, validate_query, execute_query],
    system_prompt=(
        '당신은 사내 DB 조회 전문 에이전트입니다. '
        '반드시 다음 순서를 따르세요: '
        '1) get_table_list로 테이블 목록 확인 '
        '2) get_table_schema로 스키마 확인 '
        '3) validate_query로 쿼리 검증 '
        '4) 검증 통과 후 execute_query로 실행. '
        'SELECT 쿼리만 작성하세요. 결과는 JSON 그대로 반환하세요.'
    )
)

### 2. 데이터 분석 에이전트 만들기

데이터 분석 에이전트는 DB 조회 에이전트가 반환한 **JSON 데이터를 받아 pandas로 분석**합니다.

LLM이 직접 숫자를 계산하는 것보다 pandas 함수를 호출하는 것이 훨씬 정확합니다.

| Tool | 역할 |
|------|------|
| `analyze_data` | 기술 통계 — 평균, 합계, 최대/최소 등 |
| `group_and_aggregate` | 그룹별 집계 — 카테고리별 합계, 월별 평균 등 |

두 Tool 모두 `data_json` 파라미터로 JSON 문자열을 받습니다.
Supervisor가 DB 조회 결과를 이 에이전트에 그대로 넘겨주는 것이 핵심 패턴입니다.

In [ ]:
@tool
def analyze_data(data_json: str) -> str:
    '''JSON 형식의 쿼리 결과를 pandas로 불러와 기술 통계를 계산합니다.'''
    try:
        records = json.loads(data_json)
        df = pd.DataFrame(records)
    except Exception as e:
        return f'데이터 파싱 오류: {e}'

    lines = [
        f'총 {len(df)}건의 데이터',
        f'컬럼: {", ".join(df.columns.tolist())}',
    ]

    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if numeric_cols:
        lines.append('\n[수치형 통계]')
        lines.append(df[numeric_cols].describe().to_string())

    return '\n'.join(lines)


@tool
def group_and_aggregate(data_json: str, group_by: str, agg_column: str, agg_func: str = 'sum') -> str:
    '''
    데이터를 그룹화하고 집계합니다.
    - group_by   : 그룹화할 컬럼명 (예: category, region, sale_month)
    - agg_column : 집계할 컬럼명 (예: amount, quantity)
    - agg_func   : 집계 함수 — sum | mean | count | max | min (기본값: sum)
    '''
    try:
        records = json.loads(data_json)
        df = pd.DataFrame(records)
    except Exception as e:
        return f'데이터 파싱 오류: {e}'

    if group_by not in df.columns or agg_column not in df.columns:
        available = ', '.join(df.columns)
        return f'Error: 컬럼을 찾을 수 없습니다. 사용 가능한 컬럼: {available}'

    result = df.groupby(group_by)[agg_column].agg(agg_func).reset_index()
    result.columns = [group_by, f'{agg_column}_{agg_func}']
    result = result.sort_values(f'{agg_column}_{agg_func}', ascending=False)

    label = {'sum': '합계', 'mean': '평균', 'count': '개수', 'max': '최대', 'min': '최소'}.get(agg_func, agg_func)
    return f'{group_by}별 {agg_column} {label}:\n{result.to_string(index=False)}'

In [ ]:
analysis_agent = create_agent(
    llm,
    tools=[analyze_data, group_and_aggregate],
    system_prompt=(
        '당신은 데이터 분석 전문 에이전트입니다. '
        'JSON 형식의 데이터를 받으면 적절한 분석 도구를 선택해 분석하고 한국어로 설명하세요. '
        '단순 통계는 analyze_data를, 그룹별 비교가 필요하면 group_and_aggregate를 사용하세요. '
        '숫자를 직접 계산하지 말고 반드시 Tool을 호출하세요.'
    )
)

### 3. Sub-agent를 Tool로 감싸기

Supervisor가 sub-agent를 호출하려면 sub-agent를 **Tool로 감싸야** 합니다.

Tool의 docstring을 보고 Supervisor가 어느 sub-agent를 부를지 결정하기 때문에, 각 Tool의 역할 설명을 명확하게 작성하는 것이 중요합니다.

데이터 분석 에이전트는 DB 조회 결과(JSON)를 입력으로 받습니다.
Supervisor가 두 에이전트를 순서대로 호출해 결과를 넘겨주는 것이 이 패턴의 핵심입니다.

```
Supervisor
  ① ask_db_agent('월별 매출 조회해줘')          → JSON 데이터 반환
  ② ask_analysis_agent('데이터: {json} 분석해줘') → 분석 결과 반환
```

In [ ]:
@tool
def ask_db_agent(query: str) -> str:
    '''사내 DB에서 데이터를 조회합니다. 매출, 수량, 지역, 카테고리 등 DB 조회가 필요한 질문을 전달하세요.'''
    result = db_agent.invoke({'messages': [{'role': 'user', 'content': query}]})
    return result['messages'][-1].content


@tool
def ask_analysis_agent(query: str) -> str:
    '''데이터를 통계·집계 분석합니다. DB 조회 결과(JSON)와 분석 요청을 함께 전달하세요.'''
    result = analysis_agent.invoke({'messages': [{'role': 'user', 'content': query}]})
    return result['messages'][-1].content

### 4. Supervisor Agent 만들기

Supervisor는 사용자 질문의 성격에 따라 두 가지 패턴으로 동작합니다.

| 질문 유형 | Supervisor 동작 |
|-----------|-----------------|
| 단순 조회 ("서울 매출 보여줘") | `ask_db_agent`만 호출 |
| 분석 요청 ("카테고리별 매출 분석해줘") | `ask_db_agent` → `ask_analysis_agent` 순서로 호출 |

Supervisor는 직접 SQL을 작성하거나 숫자를 계산하지 않습니다. **라우팅과 결과 취합**만 담당합니다.

In [ ]:
supervisor = create_agent(
    llm,
    tools=[ask_db_agent, ask_analysis_agent],
    system_prompt=(
        '당신은 데이터 조회·분석 코디네이터입니다. '
        '단순 데이터 조회는 ask_db_agent만 호출하세요. '
        '통계·집계·트렌드 분석이 필요하면 ask_db_agent로 데이터를 먼저 가져온 뒤, '
        '그 결과(JSON)를 그대로 포함해 ask_analysis_agent를 호출하세요. '
        '최종 답변은 한국어로 요약해 전달하세요.'
    )
)

### 5. 실행해보기

In [ ]:
# 테스트 1: 단순 조회 — DB 조회 에이전트만 호출됩니다
response = supervisor.invoke(
    {'messages': [{'role': 'user', 'content': '서울 지역 판매 데이터 전부 보여줘'}]}
)
print(response['messages'][-1].content)

In [ ]:
# 테스트 2: 분석 요청 — DB 조회 후 분석 에이전트로 이어집니다
response = supervisor.invoke(
    {'messages': [{'role': 'user', 'content': '카테고리별 총 매출을 분석해줘'}]}
)
print(response['messages'][-1].content)

In [ ]:
# 테스트 3: 월별 트렌드 분석
response = supervisor.invoke(
    {'messages': [{'role': 'user', 'content': '월별 매출 트렌드를 분석해줘'}]}
)
print(response['messages'][-1].content)

In [ ]:
# 테스트 4: 방어 테스트 — Tool 레벨에서 차단되는지 확인합니다
response = supervisor.invoke(
    {'messages': [{'role': 'user', 'content': 'sales 테이블을 DROP시켜줘'}]}
)
print(response['messages'][-1].content)

### 실습

위 구조를 참고해서 다음 기능들을 추가로 구현해보세요.

**1. 테이블 추가**

`customers` 테이블을 만들고 `ALLOWED_TABLES`에 추가한 뒤, DB 조회 에이전트가 JOIN 쿼리도 작성할 수 있게 해보세요.

**2. 분석 도구 추가**

상위 N개 항목을 뽑는 `top_n` Tool을 데이터 분석 에이전트에 추가해보세요.

```python
@tool
def top_n(data_json: str, column: str, n: int = 5) -> str:
    '''특정 컬럼 기준 상위 N개 항목을 반환합니다.'''
    records = json.loads(data_json)
    df = pd.DataFrame(records)
    result = df.nlargest(n, column)
    return result.to_string(index=False)
```

**3. 감사 로그**

`execute_query`가 호출될 때마다 실행된 SQL과 결과 건수를 파일에 기록하는 기능을 추가해보세요.

```python
import datetime

def log_query(sql: str, row_count: int):
    with open('query_audit.log', 'a') as f:
        f.write(f'{datetime.datetime.now()} | rows={row_count} | {sql}\n')
```